In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [2]:
# 1. LOAD DATA
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

Train shape: (9864, 19)
Test shape: (2466, 18)


In [3]:
# 2. CHECK MISSING VALUES

print("\nMissing values in training data:")
print(train.isnull().sum())

original_missing = train.isnull().sum().sum()

print("\nTotal missing values:", original_missing)


Missing values in training data:
Session_ID                   0
Administrative               0
Administrative_Duration    492
Informational                0
Informational_Duration       0
ProductRelated               0
ProductRelated_Duration      0
BounceRates                  0
ExitRates                  691
PageValues                   0
SpecialDay                   0
Month                        0
OperatingSystems             0
Browser                      0
Region                     591
TrafficType                789
VisitorType                394
Weekend                      0
Revenue                      0
dtype: int64

Total missing values: 2957


In [4]:
# 3. SEPARATE FEATURES AND TARGET
X = train.drop("Revenue", axis=1)
y = train["Revenue"]

In [5]:
# 4. SAVE TEST IDs
test_ids = test["Session_ID"]

In [6]:
# 5. REMOVE SESSION_ID
X = X.drop("Session_ID", axis=1)
X_test = test.drop("Session_ID", axis=1)

In [7]:
# 6. FIND NUMERICAL AND CATEGORICAL COLUMNS
numeric_columns = X.select_dtypes(
    include=["int64", "float64"]
).columns

categorical_columns = X.select_dtypes(
    include=["object", "bool"]
).columns

print("\nNumerical columns:")
print(list(numeric_columns))

print("\nCategorical columns:")
print(list(categorical_columns))


Numerical columns:
['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'OperatingSystems', 'Browser', 'Region', 'TrafficType']

Categorical columns:
['Month', 'VisitorType', 'Weekend']


In [8]:
# 7. CREATE NUMERICAL PREPROCESSING PIPELINE
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [12]:
# 8. CREATE CATEGORICAL PREPROCESSING PIPELINE
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [13]:
print("numeric_columns =", numeric_columns)
print("categorical_columns =", categorical_columns)

print("numeric_pipeline =", numeric_pipeline)
print("categorical_pipeline =", categorical_pipeline)

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_columns),
    ("cat", categorical_pipeline, categorical_columns)
])

print("Preprocessor created successfully!")

numeric_columns = Index(['Administrative', 'Administrative_Duration', 'Informational',
       'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration',
       'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay',
       'OperatingSystems', 'Browser', 'Region', 'TrafficType'],
      dtype='object')
categorical_columns = Index(['Month', 'VisitorType', 'Weekend'], dtype='object')
numeric_pipeline = Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())])
categorical_pipeline = Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')),
                ('encoder', OneHotEncoder(handle_unknown='ignore'))])
Preprocessor created successfully!


In [14]:
# 10. SPLIT DATA
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining rows:", X_train.shape[0])
print("Validation rows:", X_val.shape[0])


Training rows: 7891
Validation rows: 1973


In [15]:
# 11. FIT PREPROCESSOR ON TRAINING DATA
X_train_processed = preprocessor.fit_transform(X_train)

X_val_processed = preprocessor.transform(X_val)


print("\nProcessed training shape:",
      X_train_processed.shape)

print("Processed validation shape:",
      X_val_processed.shape)


Processed training shape: (7891, 29)
Processed validation shape: (1973, 29)


In [16]:
# 12. PREPROCESS TEST DATA
X_test_processed = preprocessor.transform(X_test)

print("Processed test shape:",
      X_test_processed.shape)

Processed test shape: (2466, 29)


In [17]:
# 13. GET PROCESSED COLUMN NAMES
processed_columns = preprocessor.get_feature_names_out()

print("\nNumber of processed columns:",
      len(processed_columns))


Number of processed columns: 29


In [ ]:
# 14. CONVERT PROCESSED DATA TO DATAFRAME
if hasattr(X_train_processed, "toarray"):
    X_train_processed_df = pd.DataFrame(
        X_train_processed.toarray(),
        columns=processed_columns
    )
else:
    X_train_processed_df = pd.DataFrame(
        X_train_processed,
        columns=processed_columns
    )


if hasattr(X_val_processed, "toarray"):
    X_val_processed_df = pd.DataFrame(
        X_val_processed.toarray(),
        columns=processed_columns
    )
else:
    X_val_processed_df = pd.DataFrame(
        X_val_processed,
        columns=processed_columns
    )


if hasattr(X_test_processed, "toarray"):
    X_test_processed_df = pd.DataFrame(
        X_test_processed.toarray(),
        columns=processed_columns
    )
else:
    X_test_processed_df = pd.DataFrame(
        X_test_processed,
        columns=processed_columns
    )


In [ ]:
# 15. ADD TARGET TO TRAINING DATA
X_train_processed_df["Revenue"] = y_train.reset_index(
    drop=True
)

X_val_processed_df["Revenue"] = y_val.reset_index(
    drop=True
)


In [20]:
# 16. SAVE PREPROCESSED DATA
X_train_processed_df.to_csv(
    "processed_train.csv",
    index=False
)

X_val_processed_df.to_csv(
    "processed_validation.csv",
    index=False
)

X_test_processed_df.to_csv(
    "processed_test.csv",
    index=False
)

print("\nPreprocessed datasets saved successfully!")


Preprocessed datasets saved successfully!


In [21]:
# 17. CREATE LOGISTIC REGRESSION MODEL

logistic_model = LogisticRegression(
    max_iter=1000
)


# ============================================================
# 18. TRAIN LOGISTIC REGRESSION
# ============================================================

logistic_model.fit(
    X_train_processed,
    y_train
)

print("\nLogistic Regression model trained successfully!")


Logistic Regression model trained successfully!


In [22]:
# 19. PREDICT VALIDATION DATA
y_pred = logistic_model.predict(
    X_val_processed
)


In [24]:
# 20. CALCULATE ACCURACY

accuracy = accuracy_score(
    y_val,
    y_pred
)

print("LOGISTIC REGRESSION RESULTS")
print("Accuracy:", accuracy)
print("Accuracy percentage:",
      accuracy * 100)

LOGISTIC REGRESSION RESULTS
Accuracy: 0.8672072985301571
Accuracy percentage: 86.72072985301571


In [25]:
# 21. CLASSIFICATION REPORT

print("\nClassification Report:")
print(
    classification_report(
        y_val,
        y_pred
    )
)


Classification Report:
              precision    recall  f1-score   support

       False       0.87      0.98      0.92      1642
        True       0.77      0.30      0.43       331

    accuracy                           0.87      1973
   macro avg       0.82      0.64      0.68      1973
weighted avg       0.86      0.87      0.84      1973



In [26]:
# 22. PREPROCESS COMPLETE TRAINING DATA

print("\nTraining final model on all training data...")

X_processed = preprocessor.fit_transform(X)

X_test_processed_final = preprocessor.transform(
    X_test
)


Training final model on all training data...


In [27]:
# 23. CREATE FINAL LOGISTIC REGRESSION MODEL
final_model = LogisticRegression(
    max_iter=1000
)

In [28]:
# 24. TRAIN FINAL MODEL
final_model.fit(
    X_processed,
    y
)

print("Final model trained successfully!")

Final model trained successfully!


In [29]:
# 25. PREDICT TEST DATA
test_predictions = final_model.predict(
    X_test_processed_final
)

print("\nNumber of test predictions:",
      len(test_predictions))


Number of test predictions: 2466


In [30]:
# 26. CREATE PREDICTION DATAFRAME

submission_predictions = pd.DataFrame({
    "id": test_ids,
    "value": test_predictions
})


In [ ]:
# 27. CONVERT PREDICTIONS TO TRUE/FALSE

submission_predictions["value"] = (
    submission_predictions["value"].astype(bool)
)

In [32]:
# 28. CHALLENGE INFORMATION


original_rows = train.shape[0]

processed_rows = X.shape[0]

original_columns = train.shape[1]

processed_columns_count = (
    len(preprocessor.get_feature_names_out())
)

row_retained_percent = (
    processed_rows / original_rows
) * 100


In [33]:
# 29. CREATE METADATA
# ============================================================

metadata = pd.DataFrame({
    "id": [
        "original_missing",
        "processed_missing",
        "original_rows",
        "processed_rows",
        "original_columns",
        "processed_columns",
        "row_retained_percent",
        "model_name"
    ],

    "value": [
        original_missing,
        0,
        original_rows,
        processed_rows,
        original_columns,
        processed_columns_count,
        row_retained_percent,
        "LogisticRegression"
    ]
})


In [ ]:
# 30. COMBINE METADATA AND PREDICTIONS

final_submission = pd.concat(
    [
        metadata,
        submission_predictions
    ],
    ignore_index=True
)


In [35]:
# 31. SAVE FINAL SUBMISSION
final_submission.to_csv(
    "submission.csv",
    index=False
)


In [36]:
# 32. DISPLAY FINAL SUBMISSION

print("FINAL SUBMISSION")
print(final_submission.head(15))

print("\nSubmission shape:",
      final_submission.shape)

print("\nFile created: submission.csv")

FINAL SUBMISSION
                      id               value
0       original_missing                2957
1      processed_missing                   0
2          original_rows                9864
3         processed_rows                9864
4       original_columns                  19
5      processed_columns                  29
6   row_retained_percent               100.0
7             model_name  LogisticRegression
8                 106094               False
9                 111845               False
10                106794                True
11                103444               False
12                106833               False
13                102684               False
14                110590               False

Submission shape: (2474, 2)

File created: submission.csv
